Tools

Models can request to call tools that perform tasks such as fetching data from a data base, searching the web, or runing code. Tools are pairings of:
1. A schema, including the name of the tools, a description, and/or argument definition (often a JSON schema)
2. A function or coroutine to execute.

Note that: LLM Hallucination, React Agent, Basic Agent

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

response=model.invoke("Write me an essay on AI")
print(response.content)


In [ ]:
# Tools

from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the wather at the location """
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])


response=model_with_tools.invoke("Whats the weather in banglore")
print(response)

response.tool_calls

### Tool execution loop

In [ ]:
# Step 1: Model generates tool cells
message=[{"role": "user", "content": "what's the weather in Bangalore?"}]
ai_msg=model_with_tools.invoke(message)
message.append(ai_msg)


# Step 2: Execute tools and collect results
for tool_cell in ai_msg.tool_cells:
    # Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_cell)
    message.append(tool_result)

# Step 3: Pass results back to model for for final response
final_response=model_with_tools.invoke(message)
print(final_response.text)